# Load modules

In [ ]:
#default_exp models.gcnn

In [ ]:
#hide
%load_ext autoreload
%autoreload 2

In [ ]:
#export
from fastai.tabular.all import *

# Message function

In [ ]:
#export
class MessageGCN(Module):
    
    """
    
    Message function from Graph convolutional network as described by Kipf & Welling (https://arxiv.org/pdf/1609.02907.pdf) and refactored as MPNN formalism.

       
    """
    
    def __init__(self, n_node_features:int, hidden_units:int, dropout:float=0.15):
        
        """
        
        Arguments:
        
        n_node_features : int
            Number of features of each atom
            
        hidden_units : int
            Number of hidden units in `self.gcn`
                       
            
        dropout : float (default = 0.15)
            Amount of dropout regularization
        
        
        """
        
        self.message_function = nn.Sequential(nn.Linear(n_node_features,hidden_units),nn.Dropout(dropout), nn.ReLU(),
                                             nn.Linear(n_node_features,hidden_units),nn.Dropout(dropout), nn.ReLU())
        
    
    def forward(self,x):
        node_features, adjacency_matrix, degree_matrix = x
        message = self.message_function(degree_matrix@adjacency_matrix@degree_matrix@node_features)
        return (message, adjacency_matrix, degree_matrix)

# Update function

In [ ]:
#export
class UpdateGCN(Module):
    
    """
    
    Vertex update function from Graph convolutional network as described by Kipf & Welling (https://arxiv.org/pdf/1609.02907.pdf) and refactored as MPNN formalism.

       
    """
    
    def __init__(self, n_node_features:int, hidden_units:int, dropout:float=0.15):
        
        """
        
        Arguments:
        
        n_node_features : int
            Number of features of each atom
            
        hidden_units : int
            Number of hidden units in `self.gcn`
            
        dropout : float (default = 0.15)
            Amount of dropout regularization
        
        
        """
        
        self.update_function = nn.Sequential(nn.Linear(n_node_features, hidden_units//2), nn.Dropout(dropout), nn.ReLU(), 
                                             nn.Linear(hidden_units//2,hidden_units//4), nn.Dropout(dropout), nn.ReLU())
    
    def forward(self, message):
        node_state = self.update_function(torch.mean(message,1))
        return node_state

# GCN model

In [ ]:
#export
class GCNMPNmodel(Module):
    
    """
    
    Prototype Graph convolutional layer as described by Kipf & Welling (https://arxiv.org/pdf/1609.02907.pdf) and refactored as MPNN formalism.

       
    """
    
    def __init__(self, n_node_features:int, hidden_units:int, num_outputs:int=1, dropout:float=0.15, y_range=None):
        
        """
        
        Arguments:
        
        n_node_features : int
            Number of features of each atom
            
        hidden_units : int
            Number of hidden units in `self.gcn`
            
        num_outputs : int
            Number of output units
                     
        dropout : float (default = 0.15)
            Amount of dropout regularization
        
        
        """
        self.message_function = nn.Sequential(MessageGCN(n_node_features, hidden_units, dropout),
                                              MessageGCN(hidden_units, hidden_units, dropout),
                                             MessageGCN(hidden_units, hidden_units,dropout))
        
        self.update_function = UpdateGCN(hidden_units, hidden_units, dropout)
        self.readout_function = nn.Linear(hidden_units, num_outputs)
        
        
        if y_range:
            self.sigmoid = SigmoidRange(*y_range)
        
    
    def forward(self, x):
        node_features, edge_features, adjacency_matrix, degree_matrix = x # Get input features 
        message = self.message_function((node_features, adjacency_matrix, degree_matrix))
        node_state = self.update_function(message[0])
        out = self.sigmoid(self.readout_function(node_state))
        return out